In [ ]:
!pwd


In [ ]:
!nvidia-smi


# Create Environment


In [ ]:
!test -d /content/edge-lipsync-model || git clone https://github.com/monkira99/edge-lipsync-model.git /content/edge-lipsync-model


In [ ]:
%cd /content/edge-lipsync-model


In [ ]:
!git pull


In [ ]:
!uv sync


In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or environment before running Hub commands."


In [ ]:
!hf auth login --token "$HF_TOKEN" --quiet

!uv run python tools/hf_model_assets.py pull \
  --repo-id tiennguyenbnbk/edge-lipsync-model-assets \
  --local-dir models


In [ ]:
!ls -lah /content/edge-lipsync-model/models/mediapipe


# Build Silent/Talking Pose-Paired Dataset Snapshot

Build the V1 dataset from `data/<persona>/silent/defaultvideo.mp4` and `data/<persona>/talking/*.mp4`.

The output snapshot is self-contained: `source_roi`, `target_roi`, and `audio [20,256]` are embedded so another machine can train with `snapshot_download()` + `load_from_disk()` without the source MP4 files.


In [ ]:
from pathlib import Path
import json
import yaml

ROOT = Path("/content/edge-lipsync-model")
PERSONA_ID = "nora"

# Preferred Colab layout. If this repo already contains data/<persona>, use it as a fallback.
DATA_ROOT = Path("/content/data")
if not (DATA_ROOT / PERSONA_ID).exists() and (ROOT / "data" / PERSONA_ID).exists():
    DATA_ROOT = ROOT / "data"

SNAPSHOT_ROOT = Path(f"/content/data/datasets/{PERSONA_ID}_pose_pairs")
WORK_ROOT = Path(f"/content/data/work/{PERSONA_ID}_pose_pairs")
HF_CACHE_DIR = Path("/content/data/hf_cache")
OUTPUT_REPO = f"tiennguyenbnbk/{PERSONA_ID}-silent-talking-pose-pairs"

WENET_ONNX = ROOT / "models/wenet/wenet.onnx"
FACE_LANDMARKER = ROOT / "models/mediapipe/face_landmarker.task"
BUILD_CONFIG_PATH = ROOT / "configs/silent_talking_dataset.colab.yaml"
PUSH_LOG = Path("/content/data/silent_talking_dataset_push.log")

for directory in (SNAPSHOT_ROOT.parent, WORK_ROOT.parent, HF_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"data_root={DATA_ROOT}")
print(f"persona_id={PERSONA_ID}")
print(f"snapshot_root={SNAPSHOT_ROOT}")
print(f"work_root={WORK_ROOT}")
print(f"output_repo={OUTPUT_REPO}")


In [ ]:
silent_video = DATA_ROOT / PERSONA_ID / "silent/defaultvideo.mp4"
talking_dir = DATA_ROOT / PERSONA_ID / "talking"
talking_videos = sorted(talking_dir.glob("*.mp4")) if talking_dir.exists() else []

assert silent_video.is_file(), f"Missing silent video: {silent_video}"
assert talking_videos, f"Missing talking videos under: {talking_dir}"
assert WENET_ONNX.is_file(), f"Missing Wenet ONNX: {WENET_ONNX}"
assert FACE_LANDMARKER.is_file(), f"Missing MediaPipe face landmarker: {FACE_LANDMARKER}"

print(f"silent={silent_video}")
print(f"talking_videos={len(talking_videos)}")
for path in talking_videos[:10]:
    print(f"- {path.name}")


In [ ]:
build_config = {
    "data_root": str(DATA_ROOT),
    "persona_id": PERSONA_ID,
    "snapshot_root": str(SNAPSHOT_ROOT),
    "work_root": str(WORK_ROOT),
    "wenet_onnx": str(WENET_ONNX),
    "landmark_model_asset_path": str(FACE_LANDMARKER),
    "fps": 25,
    "sample_rate": 16000,
    "validation_fraction": 0.20,
    "split_salt": "edge-lipsync-silent-talking-v1",
    "idle_max_ratio": 0.10,
    "idle_sample_weight": 0.25,
    "preview_count_per_group": 8,
    "progress": False,
    "strict": True,
    "matching": {
        "max_yaw_delta": 5.0,
        "max_pitch_delta": 5.0,
        "max_roll_delta": 4.0,
        "max_center_x_delta": 0.05,
        "max_center_y_delta": 0.05,
        "min_width_ratio": 0.9,
        "max_width_ratio": 1.1,
        "min_height_ratio": 0.9,
        "max_height_ratio": 1.1,
        "pose_weight": 1.0,
        "position_weight": 1.0,
        "scale_weight": 1.0,
    },
    "post_crop_alignment": {
        "max_stable_landmark_rmse": 0.04,
        "max_mouth_center_delta": 0.04,
    },
    "blur": {
        "min_source_face_laplacian_variance": 60.0,
        "min_target_face_laplacian_variance": 60.0,
        "min_target_mouth_laplacian_variance": 40.0,
    },
    "sync": {
        "window_seconds": 2.0,
        "stride_seconds": 1.0,
        "max_lag_frames": 3,
        "max_reject_lag_frames": 2,
        "min_correlation": 0.20,
        "silence_rms_threshold": 0.001,
        "speech_fraction_threshold": 0.25,
    },
}

BUILD_CONFIG_PATH.write_text(yaml.safe_dump(build_config, sort_keys=False), encoding="utf-8")
print(BUILD_CONFIG_PATH)


In [ ]:
!sed -n "1,240p" {BUILD_CONFIG_PATH}


In [ ]:
!uv run python tools/build_silent_talking_dataset.py \
  --config {BUILD_CONFIG_PATH} \
  --strict \
  --no-progress


In [ ]:
from datasets import Image, load_from_disk

metadata = json.loads((SNAPSHOT_ROOT / "build_complete.json").read_text())
dataset = load_from_disk(SNAPSHOT_ROOT / "dataset")
raw_roi = dataset["train"].cast_column("source_roi", Image(decode=False))[0]["source_roi"]

print(json.dumps({
    "schema_version": metadata["schema_version"],
    "train_rows": metadata["train_rows"],
    "val_rows": metadata["val_rows"],
    "talking_clips": metadata["talking_clips"],
    "failed_clips": metadata["failed_clips"],
    "split_mode": metadata["split_mode"],
    "dataset_fingerprints": metadata["dataset_fingerprints"],
}, indent=2))
print(dataset)
print(f"source_roi_path={raw_roi['path']}")
print(f"source_roi_png_magic={raw_roi['bytes'][:8]}")


# Review Sample Pairs Before Upload

Run these cells before `push-snapshot`. They show both aggregate rejection stats and visual examples so you can catch bad pose matching, ROI alignment drift, blur, wrong crop scale, or suspicious sync before publishing the dataset.


In [ ]:
from collections import Counter

quality_dir = SNAPSHOT_ROOT / "reports/quality"
clip_reports = sorted(
    path for path in quality_dir.glob("*.json")
    if path.name != "silent.json" and not path.name.endswith("_frame_decisions.json")
)

print(f"quality_dir={quality_dir}")
print(f"clip_reports={len(clip_reports)}")
for path in clip_reports:
    report = json.loads(path.read_text())
    print("=" * 96)
    print(path.name)
    print(json.dumps({
        "status": report.get("status"),
        "frame_count": report.get("frame_count"),
        "valid_analysis_count": report.get("valid_analysis_count"),
        "speech_pair_count": report.get("speech_pair_count"),
        "retained_idle_pair_count": report.get("retained_idle_pair_count"),
        "rejection_counts": report.get("rejection_counts"),
        "sync": report.get("sync"),
        "matching_score": report.get("matching_score"),
        "valid_silent_candidate_count": report.get("valid_silent_candidate_count"),
        "post_crop_alignment_mismatch_count": report.get("post_crop_alignment_mismatch_count"),
        "post_crop_alignment_mismatch_mouth_center": report.get("post_crop_alignment_mismatch_mouth_center"),
    }, indent=2))


In [ ]:
from IPython.display import Image as IPyImage, Markdown, display

PREVIEW_GROUPS = [
    "best_score",
    "near_pose_threshold",
    "near_center_threshold",
    "near_scale_threshold",
    "near_post_crop_threshold",
    "smallest_margin",
    "low_sync_confidence",
    "idle_retained",
]
PREVIEW_COUNT = 4

previews_root = SNAPSHOT_ROOT / "reports/previews"
assert previews_root.is_dir(), f"Missing preview directory: {previews_root}"

for clip_dir in sorted(path for path in previews_root.iterdir() if path.is_dir()):
    display(Markdown(f"## Preview clip: `{clip_dir.name}`"))
    for group in PREVIEW_GROUPS:
        images = sorted((clip_dir / group).glob("*.png"))[:PREVIEW_COUNT]
        if not images:
            continue
        display(Markdown(f"### `{group}`"))
        for image_path in images:
            display(IPyImage(filename=str(image_path), width=900))

    rejected_root = clip_dir / "rejected"
    if rejected_root.is_dir():
        display(Markdown("### `rejected examples`"))
        for reason_dir in sorted(path for path in rejected_root.iterdir() if path.is_dir()):
            images = sorted(reason_dir.glob("*.png"))[:PREVIEW_COUNT]
            if not images:
                continue
            display(Markdown(f"#### `{reason_dir.name}`"))
            for image_path in images:
                display(IPyImage(filename=str(image_path), width=900))


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

MAX_DATASET_SAMPLES = 10


def row_sort_key(row):
    return (str(row["talking_clip_id"]), int(row["target_frame_idx"]), int(row["source_frame_idx"]))


def rows_from_split(split: str) -> list[dict]:
    return [dict(row, split=split) for row in dataset[split]]


all_rows = rows_from_split("train") + rows_from_split("val")
all_rows = sorted(all_rows, key=row_sort_key)

candidates: list[tuple[str, dict]] = []
for split in ("train", "val"):
    split_rows = rows_from_split(split)
    if split_rows:
        candidates.append((f"{split}_first", split_rows[0]))

idle_rows = [row for row in all_rows if row.get("is_idle")]
if idle_rows:
    candidates.append(("idle_retained", idle_rows[len(idle_rows) // 2]))

low_sync_rows = [row for row in all_rows if row.get("sync_confidence") == "low"]
if low_sync_rows:
    candidates.append(("low_sync_confidence", low_sync_rows[len(low_sync_rows) // 2]))

if all_rows:
    candidates.append(("highest_matching_score", max(all_rows, key=lambda row: float(row["matching_score"]))))
    candidates.append(("largest_post_crop_alignment", max(
        all_rows,
        key=lambda row: max(
            float(row["stable_landmark_alignment_rmse"]),
            float(row["mouth_center_delta_after_crop"]),
        ),
    )))
    candidates.append(("fewest_valid_silent_candidates", min(
        all_rows,
        key=lambda row: int(row["valid_silent_candidate_count"]),
    )))

seen_pair_ids: set[str] = set()
unique_candidates: list[tuple[str, dict]] = []
for label, row in candidates:
    pair_id = str(row["pair_id"])
    if pair_id in seen_pair_ids:
        continue
    seen_pair_ids.add(pair_id)
    unique_candidates.append((label, row))


def make_pair_panel(label: str, row: dict) -> Image.Image:
    source = row["source_roi"].convert("RGB")
    target = row["target_roi"].convert("RGB")
    width = source.width + target.width + 420
    height = max(source.height, target.height, 230)
    panel = Image.new("RGB", (width, height), "white")
    panel.paste(source, (0, 0))
    panel.paste(target, (source.width, 0))
    draw = ImageDraw.Draw(panel)
    x = source.width + target.width + 12
    lines = [
        label,
        f"pair={row['pair_id']}",
        f"split={row['split']} idle={row['is_idle']} weight={float(row['sample_weight']):.2f}",
        f"src={row['source_frame_idx']} tgt={row['target_frame_idx']} audio={row['audio_idx']}",
        f"score={float(row['matching_score']):.3f} candidates={row['valid_silent_candidate_count']}",
        f"pose=({float(row['pose_delta_yaw']):.2f}, {float(row['pose_delta_pitch']):.2f}, {float(row['pose_delta_roll']):.2f})",
        f"center=({float(row['center_delta_x']):.4f}, {float(row['center_delta_y']):.4f})",
        f"scale=({float(row['width_ratio']):.3f}, {float(row['height_ratio']):.3f})",
        f"align=({float(row['stable_landmark_alignment_rmse']):.4f}, {float(row['mouth_center_delta_after_crop']):.4f})",
        f"sync={row['sync_best_lag_frames']} corr={float(row['sync_correlation']):.3f} {row['sync_confidence']}",
        f"flags={list(row['flags'])}",
    ]
    y = 8
    for text in lines:
        draw.text((x, y), text[:72], fill=(20, 20, 20))
        y += 18
    return panel


print("Left image = source silent ROI. Middle image = target talking ROI. Text panel = gate/debug metadata.")
for label, row in unique_candidates[:MAX_DATASET_SAMPLES]:
    display(make_pair_panel(label, row))


In [ ]:
!uv run python tools/hf_dataset.py push-snapshot \
  --snapshot-root {SNAPSHOT_ROOT} \
  --repo-id {OUTPUT_REPO} | tee {PUSH_LOG}


In [ ]:
push_lines = PUSH_LOG.read_text(encoding="utf-8").splitlines()
revision_line = next(line for line in push_lines if line.startswith("revision="))
DATASET_REVISION = revision_line.split("=", 1)[1].strip()
LOCAL_SNAPSHOT_ROOT = Path("/content/data/hf_snapshots") / OUTPUT_REPO.split("/")[-1] / DATASET_REVISION

print(f"HF_DATASET_REPO = {OUTPUT_REPO!r}")
print(f"HF_DATASET_REVISION = {DATASET_REVISION!r}")
print(f"HF_DATASET_LOCAL_DIR = {str(LOCAL_SNAPSHOT_ROOT)!r}")


In [ ]:
!uv run python tools/hf_dataset.py pull-snapshot \
  --repo-id {OUTPUT_REPO} \
  --revision {DATASET_REVISION} \
  --local-dir {LOCAL_SNAPSHOT_ROOT} \
  --cache-dir {HF_CACHE_DIR}


In [ ]:
print("Copy these values into notebooks/training_dev.ipynb:")
print(f"HF_DATASET_REPO = {OUTPUT_REPO!r}")
print(f"HF_DATASET_REVISION = {DATASET_REVISION!r}")
print(f"HF_DATASET_LOCAL_DIR = {str(LOCAL_SNAPSHOT_ROOT)!r}")
